In [1]:
from langgraph.graph import StateGraph, END, START
from langgraph.graph.message import add_messages

from langchain_core.messages import HumanMessage, SystemMessage, BaseMessage
from langchain_ollama import ChatOllama
from typing import TypedDict, Annotated

from langgraph.checkpoint.memory import MemorySaver


In [2]:
class ChatState(TypedDict):

    messages: Annotated[list[BaseMessage], add_messages]

In [3]:
llm = ChatOllama(model= "llama3.2:1b")

In [4]:
def chat_node(state: ChatState):
    messages = state['messages']
    response = llm.invoke(messages)
    return {"messages": [response]}

In [5]:
checkPointer = MemorySaver()
graph = StateGraph(ChatState)

graph.add_node("chat_node", chat_node)

graph.add_edge(START, "chat_node")
graph.add_edge("chat_node", END)

workflow = graph.compile(checkpointer=checkPointer)

In [6]:
initial_state = {
    'messages': [HumanMessage(content='What is the capital of india')]
}

workflow.invoke(initial_state)['messages'][-1].content

ValueError: Checkpointer requires one or more of the following 'configurable' keys: thread_id, checkpoint_ns, checkpoint_id

In [7]:
thread_id = "1"

while True:
    user_input = input("User: ")
    print(f"User: {user_input}")

    if user_input.strip().lower() in ["exit", "close", "bye"]:
        break

    config = {'configurable' : {'thread_id' : thread_id}}

    response = workflow.invoke({
        'messages': [HumanMessage(content=user_input)]
    }, config=config)

    print(f"AI: {response["messages"][-1].content}")

User: what is my name
AI: This conversation has just started. I'm not aware of any information about your name, and I don't have any context to provide. I'm here to help answer any questions you may have, though. What's on your mind?
User: yes
AI: I want to make sure I'm respecting your boundaries. Since we just started our conversation, I don't have any prior information about you, including your name. If you'd like to share your name with me, I'd be happy to use it in our conversation. Alternatively, if you'd rather not share your name, that's okay too. I can simply refer to you as "user" or "visitor" instead.
User: bye


In [ ]:
workflow.get_state(config=config)

StateSnapshot(values={'messages': [HumanMessage(content='Hi my name is aakash', additional_kwargs={}, response_metadata={}, id='e5ee5fff-9a98-4de7-b8c7-4297ba55ddc6'), AIMessage(content="Hello Akash! It's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'model': 'llama3.2:1b', 'created_at': '2026-11-02T22:25:53.744845087Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1510191456, 'load_duration': 1390760873, 'prompt_eval_count': 31, 'prompt_eval_duration': 20638000, 'eval_count': 27, 'eval_duration': 97702000, 'logprobs': None, 'model_name': 'llama3.2:1b', 'model_provider': 'ollama'}, id='lc_run--01a19e5e-726a-7652-b0c2-7087275781fb-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 31, 'output_tokens': 27, 'total_tokens': 58}), HumanMessage(content='what is my name', additional_kwargs={}, response_metadata={}, id='105655b1-2649-49ea-be61-4c2a4f619e52'), AIMessage(content='I r